# 02 - SGI Exploration

This notebook explores the Spectral Geometry Instability (SGI) signal.

## Steps:
1. Compute rolling covariance matrices
2. Compute SGI and related metrics
3. Visualize SGI over time
4. Overlay crisis periods
5. Compare with baseline metrics

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.config import load_experiment_config
from src.data_loader import load_or_download_prices
from src.preprocessing import preprocess_data
from src.covariance import rolling_covariance_matrices
from src.spectral_geometry import compute_rolling_geometry_features, compute_sgi_derived_features
from src.features import build_feature_dataframe
from src.plots import plot_sgi_time_series, plot_sgi_vs_volatility, plot_sgi_vs_correlation
from src.constants import CRISIS_DATES
from src.paths import ensure_all_directories

ensure_all_directories()

## Load and Preprocess Data

In [ ]:
config = load_experiment_config('../configs/universe_sector_etfs.yaml')

tickers = config['universe']['tickers']
prices = load_or_download_prices(tickers, config['data']['start_date'], config['data']['end_date'])
prices, returns, _ = preprocess_data(prices, return_method='log', save_outputs=False)

print(f"Returns shape: {returns.shape}")

## Compute Rolling Covariance

In [ ]:
rolling_window = config['covariance']['rolling_window']
estimator = config['covariance']['estimator']

print(f"Computing {estimator} covariance with window={rolling_window}...")
cov_matrices = rolling_covariance_matrices(returns, rolling_window, method=estimator)
print(f"Computed {len(cov_matrices)} covariance matrices")

## Compute SGI Metrics

In [ ]:
top_k = config['geometry']['top_k']

geometry_df = compute_rolling_geometry_features(cov_matrices, top_k)
geometry_df = compute_sgi_derived_features(geometry_df)

print(f"Geometry features shape: {geometry_df.shape}")
print(f"Columns: {list(geometry_df.columns)}")
geometry_df.head()

In [ ]:
# SGI summary statistics
print("SGI Summary Statistics:")
print(geometry_df[['sgi', 'weighted_sgi', 'max_angle', 'absorption_ratio']].describe())

## Visualize SGI Over Time

In [ ]:
plot_sgi_time_series(geometry_df, CRISIS_DATES)

## Compare SGI with Baseline Metrics

In [ ]:
features_df = build_feature_dataframe(returns, geometry_df, config, prices)
print(f"Full features shape: {features_df.shape}")

In [ ]:
plot_sgi_vs_volatility(features_df)

In [ ]:
plot_sgi_vs_correlation(features_df)

In [ ]:
# Correlation matrix of key features
key_features = ['sgi', 'weighted_sgi', 'realized_vol', 'avg_correlation', 'absorption_ratio']
available = [f for f in key_features if f in features_df.columns]

corr_matrix = features_df[available].corr()
print("Feature Correlations:")
print(corr_matrix.round(3))

## Key Observations

1. SGI spikes around major crisis periods
2. SGI is correlated with but distinct from volatility
3. SGI captures geometric instability beyond simple correlation changes